# Qversity Fintech — Análisis de Preguntas de Negocio Clave

**Pipeline:** ELT end-to-end migrado a Databricks Free Edition  
**Stack:** PySpark (bronze → silver_raw) · dbt-databricks (silver → gold) · Unity Catalog  
**Dataset:** ~5.100 clientes sintéticos, 7 países LATAM, 87.686 transacciones  

Este notebook analiza **una pregunta de negocio por cada página del dashboard de Power BI**:

| # | Dashboard page | Pregunta |
|---|---|---|
| 1 | Revenue & Transactions | ¿Qué canales concentran más volumen y cuál es su confiabilidad operacional? |
| 2 | Risk & Credit | ¿En qué segmento de clientes se concentra el riesgo de delinquencia y default? |
| 3 | Customer & Engagement | ¿Existe estacionalidad o tendencia en la adquisición mensual de clientes? |

---
> **Fuente de datos:** `workspace.gold.*` (Unity Catalog, Databricks Free Edition)  
> Las limitaciones del dataset sintético se documentan en cada sección y en `docs/decisions.md`.

In [ ]:
CATALOG = "workspace"
spark.sql(f"USE CATALOG {CATALOG}")
print(f"Usando catálogo: {CATALOG}")
print(f"Schema de trabajo: gold")

---
# Pregunta 1 — Revenue & Transactions
## ¿Qué canales concentran más volumen transaccional y cuál es su confiabilidad operacional?

### Contexto de negocio

En una institución financiera LATAM, los canales de transacción (mobile, web, ATM,
branch, POS) no solo difieren en volumen sino en **perfil de riesgo operacional**.
Un canal de alto volumen con alta tasa de fallo implica:
- Pérdida de ingresos por transacciones no completadas
- Fricción en la experiencia del cliente → churn
- Costos operativos de resolución de disputas

La pregunta estratégica no es "¿qué canal mueve más plata?" sino
**"¿qué canal mueve más plata con menor riesgo de fallo?"**

### Fuente
**Mart:** `gold.mart_tx_by_channel` · **Grain:** (channel, currency)  
**Columnas clave:** `tx_count`, `completed_tx_count`, `failed_tx_count`, `total_value`, `avg_ticket`, `failed_rate`

### 1.1 — Desempeño por canal en USD (moneda de referencia)

In [ ]:
df_channel_usd = spark.sql("""
    SELECT
        channel,
        tx_count,
        completed_tx_count,
        failed_tx_count,
        ROUND(total_value, 2)               AS total_value_usd,
        ROUND(avg_ticket, 2)                AS avg_ticket_usd,
        ROUND(failed_rate * 100, 2)         AS failed_rate_pct,
        ROUND(total_value * failed_rate, 2) AS value_at_risk_usd,
        ROUND((1 - failed_rate) * 100, 2)   AS completion_rate_pct
    FROM gold.mart_tx_by_channel
    WHERE currency = 'USD'
    ORDER BY total_value DESC
""")
display(df_channel_usd)

### 1.2 — Resumen global por canal (todas las monedas agregadas)

In [ ]:
df_channel_global = spark.sql("""
    SELECT
        channel,
        SUM(tx_count)                               AS total_tx,
        SUM(completed_tx_count)                     AS total_completed,
        SUM(failed_tx_count)                        AS total_failed,
        ROUND(SUM(failed_tx_count) * 100.0
              / NULLIF(SUM(tx_count), 0), 2)        AS global_failed_rate_pct,
        ROUND(SUM(completed_tx_count) * 100.0
              / NULLIF(SUM(tx_count), 0), 2)        AS global_completion_rate_pct,
        ROUND(AVG(avg_ticket), 2)                   AS avg_ticket_all_currencies,
        ROUND(SUM(completed_tx_count) * 1.0
              / NULLIF(SUM(tx_count), 0)
              * SUM(tx_count), 0)                   AS effective_volume_score
    FROM gold.mart_tx_by_channel
    GROUP BY channel
    ORDER BY effective_volume_score DESC
""")
display(df_channel_global)

### 1.3 — Top 15 categorías por valor (USD)

In [ ]:
df_category = spark.sql("""
    SELECT
        category,
        currency,
        tx_count,
        completed_tx_count,
        ROUND(total_value, 2)                                        AS total_value,
        ROUND(avg_ticket, 2)                                         AS avg_ticket,
        ROUND(completed_tx_count * 100.0 / NULLIF(tx_count, 0), 1)  AS completion_rate_pct
    FROM gold.mart_tx_by_category
    WHERE currency = 'USD'
    ORDER BY total_value DESC
    LIMIT 15
""")
display(df_category)

### 1.4 — Volumen por día de la semana (¿hay picos operacionales?)

In [ ]:
df_dow = spark.sql("""
    SELECT
        day_of_week,
        day_of_week_name,
        SUM(tx_count)              AS total_tx,
        SUM(completed_tx_count)    AS total_completed,
        ROUND(SUM(total_value), 2) AS total_value_usd,
        ROUND(AVG(avg_ticket), 2)  AS avg_ticket_usd
    FROM gold.mart_tx_by_dow
    WHERE currency = 'USD'
    GROUP BY day_of_week, day_of_week_name
    ORDER BY day_of_week
""")
display(df_dow)

### 1.5 — Top 10 merchants por valor (USD)

In [ ]:
df_merchants = spark.sql("""
    SELECT
        merchant,
        top_category,
        tx_count,
        ROUND(total_value, 2)              AS total_value_usd,
        ROUND(avg_ticket, 2)               AS avg_ticket_usd,
        rank_by_value_within_currency      AS rank_usd
    FROM gold.mart_top_merchants
    WHERE currency = 'USD'
      AND rank_by_value_within_currency <= 10
    ORDER BY rank_by_value_within_currency
""")
display(df_merchants)

### Hallazgos — Revenue & Transactions

**1. Tasa de fallo ~25% en todos los canales:**  
La tasa de transacciones fallidas ronda el 25% en todos los canales,
muy por encima del benchmark real de la industria (< 3%). El generador
sintético distribuyó los estados de transacción uniformemente entre
`completed / pending / failed / reversed` (~25% cada uno), sin el sesgo
realista donde > 95% serían `completed`. Este hallazgo está documentado
en `decisions.md` y es intencionalmente preservado — no se enmascara el dato fuente.

**2. Volúmenes muy homogéneos entre canales:**  
Los 5 canales muestran ~17.5K transacciones cada uno. En una operación real,
`mobile` y `web` concentrarían proporcionalmente más volumen que `branch` o
`atm` por el crecimiento del canal digital en LATAM. La homogeneidad es un
artefacto del generador que no correlacionó el canal con el segmento de cliente.

**3. Distribución por día de la semana plana:**  
No hay picos operacionales claros. En una banca real, lunes y viernes
concentrarían más transacciones por comportamientos de nómina y pagos de fin
de semana. La distribución plana confirma que las fechas fueron generadas aleatoriamente.

**Recomendación operativa (contexto real):** Priorizar la investigación de
fallos en los canales de mayor ticket promedio — un fallo en `branch` con
avg_ticket alto impacta más revenue que en `atm` con ticket bajo.

---
# Pregunta 2 — Risk & Credit
## ¿En qué segmento de clientes se concentra el riesgo de delinquencia y default?

### Contexto de negocio

La gestión del riesgo crediticio en una institución LATAM requiere segmentar
la cartera para identificar dónde concentrar recursos de cobranza, dónde
endurecer los criterios de originación y dónde existe potencial de cross-sell
sin aumentar el riesgo sistémico.

Las métricas clave son:
- **Delinquency rate:** % de clientes con días pasados de vencimiento (DPD) ≥ 30
- **Default rate:** % de clientes con DPD ≥ 90 o status = 'default' (Basel/IFRS9)
- **Credit score distribution:** proxy de calidad crediticia inicial
- **Utilization vs delinquency:** si la utilización predice el fallo (hipótesis FICO)

### Fuente
**Marts:** `gold.mart_delinquency_by_segment` · `gold.mart_customer_360`  
`gold.mart_credit_score_by_country` · `gold.mart_utilization_vs_delinquency`  
`gold.mart_loan_dpd` · `gold.mart_risk_buckets`

### 2.1 — Delinquencia y default por segmento de cliente

In [ ]:
df_delinquency = spark.sql("""
    SELECT
        d.customer_segment,
        d.customer_count,
        d.delinquent_customers,
        ROUND(d.delinquency_rate * 100, 2)  AS delinquency_rate_pct,
        ROUND(d.default_rate * 100, 2)      AS default_rate_pct,
        COUNT(CASE WHEN c.credit_score < 580
                    AND c.credit_score IS NOT NULL
                   THEN 1 END)              AS poor_credit_count,
        ROUND(
            COUNT(CASE WHEN c.credit_score < 580
                        AND c.credit_score IS NOT NULL
                       THEN 1 END) * 100.0
            / NULLIF(d.customer_count, 0), 1
        )                                   AS pct_poor_credit,
        ROUND(AVG(c.credit_score), 0)       AS avg_credit_score
    FROM gold.mart_delinquency_by_segment d
    LEFT JOIN gold.mart_customer_360 c
        ON c.customer_segment = d.customer_segment
    GROUP BY
        d.customer_segment,
        d.customer_count,
        d.delinquent_customers,
        d.delinquency_rate,
        d.default_rate
    ORDER BY d.delinquency_rate DESC
""")
display(df_delinquency)

### 2.2 — Distribución de credit score buckets por segmento

In [ ]:
df_score_segment = spark.sql("""
    SELECT
        customer_segment,
        credit_score_bucket,
        COUNT(*)    AS customer_count,
        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (PARTITION BY customer_segment), 1
        )           AS pct_within_segment
    FROM gold.mart_customer_360
    WHERE credit_score_bucket IS NOT NULL
    GROUP BY customer_segment, credit_score_bucket
    ORDER BY customer_segment,
             CASE credit_score_bucket
                 WHEN 'exceptional' THEN 1
                 WHEN 'very_good'   THEN 2
                 WHEN 'good'        THEN 3
                 WHEN 'fair'        THEN 4
                 WHEN 'poor'        THEN 5
                 WHEN 'unknown'     THEN 6
                 ELSE 7
             END
""")
display(df_score_segment)

### 2.3 — Distribución de credit score por país

In [ ]:
df_score_country = spark.sql("""
    SELECT
        country,
        credit_score_bucket,
        customer_count,
        ROUND(bucket_share_within_country * 100, 1) AS pct_within_country
    FROM gold.mart_credit_score_by_country
    ORDER BY country,
             CASE credit_score_bucket
                 WHEN 'exceptional' THEN 1
                 WHEN 'very_good'   THEN 2
                 WHEN 'good'        THEN 3
                 WHEN 'fair'        THEN 4
                 WHEN 'poor'        THEN 5
                 WHEN 'unknown'     THEN 6
                 ELSE 7
             END
""")
display(df_score_country)

### 2.4 — Utilización vs delinquencia (¿la hipótesis FICO se sostiene?)

In [ ]:
df_utilization = spark.sql("""
    SELECT
        utilization_bucket,
        customer_count,
        ROUND(delinquency_rate * 100, 1) AS delinquency_rate_pct,
        ROUND(avg_credit_score, 0)       AS avg_credit_score
    FROM gold.mart_utilization_vs_delinquency
    ORDER BY
        CASE utilization_bucket
            WHEN 'healthy'   THEN 1
            WHEN 'moderate'  THEN 2
            WHEN 'high'      THEN 3
            WHEN 'maxed'     THEN 4
            ELSE 5
        END
""")
display(df_utilization)

### 2.5 — DPD aging por tipo de préstamo (USD)

In [ ]:
df_dpd = spark.sql("""
    SELECT
        loan_type,
        dpd_bucket,
        SUM(loan_count)                    AS loan_count,
        ROUND(SUM(outstanding_balance), 2) AS outstanding_balance_usd,
        ROUND(AVG(avg_dpd), 1)             AS avg_dpd,
        ROUND(AVG(bucket_share) * 100, 1)  AS avg_bucket_share_pct
    FROM gold.mart_loan_dpd
    WHERE currency = 'USD'
    GROUP BY loan_type, dpd_bucket
    ORDER BY loan_type, dpd_bucket
""")
display(df_dpd)

### 2.6 — Segmentación por risk buckets

In [ ]:
df_risk = spark.sql("""
    SELECT
        risk_bucket,
        customer_count,
        ROUND(delinquency_rate * 100, 1) AS delinquency_rate_pct,
        ROUND(default_rate * 100, 1)     AS default_rate_pct
    FROM gold.mart_risk_buckets
    ORDER BY
        CASE risk_bucket
            WHEN 'low'      THEN 1
            WHEN 'medium'   THEN 2
            WHEN 'high'     THEN 3
            WHEN 'critical' THEN 4
            ELSE 5
        END
""")
display(df_risk)

### Hallazgos — Risk & Credit

**1. Delinquencia ~63% y default ~25% uniformes entre segmentos:**  
En una cartera real, `retail` mostraría delinquencia 2-3x mayor que `private_banking`.
La uniformidad es un artefacto del generador que no correlacionó el estado del
préstamo con el segmento. Documentado en `decisions.md`.

**2. 46% de clientes en banda Poor (score < 580):**  
Con un score promedio de 573, el portfolio está sesgado hacia la banda de mayor
riesgo crediticio. Hallazgo consistente con el EDA del día 1, refleja un sesgo
del generador, no una selección adversa real.

**3. La hipótesis FICO no se sostiene:**  
La utilización (`healthy → maxed`) no predice la delinquencia — todos los buckets
muestran la misma tasa (~63%). En una cartera real, los clientes `maxed` tienen
delinquencia 4-5x mayor que los `healthy`. La ausencia de correlación confirma
que el generador no modeló las relaciones causales entre variables financieras.

**4. 41% del balance outstanding en DPD 180+:**  
Concentración extrema en el bucket más severo — indicaría write-offs inevitables
en una cartera real. Documentado en `decisions.md` como anomalía de datos sintéticos.

**Recomendación (contexto real):** El indicador más útil para targeting de
cobranza en este dataset es el `credit_score_bucket` combinado con `days_past_due`,
no el segmento de cliente por sí solo.

---
# Pregunta 3 — Customer & Engagement
## ¿Existe estacionalidad o tendencia en la adquisición mensual de clientes?

### Contexto de negocio

Entender si la adquisición de clientes tiene patrones estacionales permite
a la institución:
- **Optimizar el presupuesto de marketing:** concentrar inversión en meses
  de alta demanda natural en vez de pelear contra la estacionalidad
- **Dimensionar recursos de onboarding:** anticipar picos de nuevas altas
- **Evaluar efectividad de campañas:** aislar el efecto campaña del efecto estacional base

La pregunta específica es: **¿hay meses consistentemente mejores o peores
para adquirir clientes nuevos, o la adquisición es aleatoria?**

### Fuente
**Mart:** `gold.mart_acquisition_trend` · **Grain:** mes calendario  
**Columnas clave:** `month`, `new_customers`, `cumulative_customers`, `mom_growth_pct`

### 3.1 — Tendencia completa con promedio móvil 3 meses (2020-2026)

In [ ]:
df_trend_full = spark.sql("""
    SELECT
        month_label,
        year,
        month_number,
        new_customers,
        cumulative_customers,
        ROUND(mom_growth_pct, 2)    AS mom_growth_pct,
        ROUND(AVG(new_customers) OVER (
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1)                       AS rolling_avg_3m
    FROM gold.mart_acquisition_trend
    ORDER BY month
""")
display(df_trend_full)

### 3.2 — Estacionalidad: promedio, mín, máx y stddev por mes del año

In [ ]:
df_seasonality = spark.sql("""
    SELECT
        month_number,
        CASE month_number
            WHEN 1  THEN '01 - Enero'
            WHEN 2  THEN '02 - Febrero'
            WHEN 3  THEN '03 - Marzo'
            WHEN 4  THEN '04 - Abril'
            WHEN 5  THEN '05 - Mayo'
            WHEN 6  THEN '06 - Junio'
            WHEN 7  THEN '07 - Julio'
            WHEN 8  THEN '08 - Agosto'
            WHEN 9  THEN '09 - Septiembre'
            WHEN 10 THEN '10 - Octubre'
            WHEN 11 THEN '11 - Noviembre'
            WHEN 12 THEN '12 - Diciembre'
        END                             AS month_name,
        COUNT(*)                        AS years_observed,
        ROUND(AVG(new_customers), 1)    AS avg_new_customers,
        MIN(new_customers)              AS min_new_customers,
        MAX(new_customers)              AS max_new_customers,
        MAX(new_customers) - MIN(new_customers) AS range_customers,
        ROUND(STDDEV(new_customers), 1) AS stddev_customers
    FROM gold.mart_acquisition_trend
    -- Excluir mayo 2026: mes incompleto (17 clientes = artefacto de corte)
    WHERE NOT (year = 2026 AND month_number = 5)
    GROUP BY month_number
    ORDER BY month_number
""")
display(df_seasonality)

### 3.3 — Distribución del crecimiento MoM en 5 buckets

In [ ]:
df_growth_dist = spark.sql("""
    SELECT
        CASE
            WHEN mom_growth_pct > 20   THEN '5. Crecimiento fuerte   (> +20%)'
            WHEN mom_growth_pct > 5    THEN '4. Crecimiento moderado (+5% a +20%)'
            WHEN mom_growth_pct >= -5  THEN '3. Estable              (-5% a +5%)'
            WHEN mom_growth_pct >= -20 THEN '2. Caída moderada       (-20% a -5%)'
            ELSE                            '1. Caída fuerte         (< -20%)'
        END                             AS growth_bucket,
        COUNT(*)                        AS months_count,
        ROUND(AVG(mom_growth_pct), 1)   AS avg_growth_pct,
        MIN(mom_growth_pct)             AS min_growth_pct,
        MAX(mom_growth_pct)             AS max_growth_pct
    FROM gold.mart_acquisition_trend
    WHERE mom_growth_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""")
display(df_growth_dist)

### 3.4 — Adquisición por año: ¿hay crecimiento interanual?

In [ ]:
df_yearly = spark.sql("""
    SELECT
        year,
        SUM(new_customers)           AS total_new_customers,
        ROUND(AVG(new_customers), 1) AS avg_per_month,
        MIN(new_customers)           AS min_month,
        MAX(new_customers)           AS max_month,
        COUNT(*)                     AS months_with_data
    FROM gold.mart_acquisition_trend
    WHERE NOT (year = 2026 AND month_number = 5)
    GROUP BY year
    ORDER BY year
""")
display(df_yearly)

### 3.5 — Adopción digital por segmento de cliente

In [ ]:
df_digital = spark.sql("""
    SELECT
        customer_segment,
        customer_count,
        ROUND(mobile_adoption_rate * 100, 1)      AS mobile_adoption_pct,
        ROUND(web_adoption_rate * 100, 1)         AS web_adoption_pct,
        ROUND(any_digital_adoption_rate * 100, 1) AS any_digital_pct
    FROM gold.mart_digital_adoption_by_segment
    ORDER BY any_digital_adoption_rate DESC
""")
display(df_digital)

### 3.6 — Preferencia de canal por grupo de edad

In [ ]:
df_channel_age = spark.sql("""
    SELECT
        age_bucket,
        preferred_channel,
        customer_count,
        ROUND(bucket_share * 100, 1) AS pct_within_age_bucket
    FROM gold.mart_channel_preference_by_age
    ORDER BY
        CASE age_bucket
            WHEN '18-25' THEN 1
            WHEN '26-35' THEN 2
            WHEN '36-50' THEN 3
            WHEN '51-65' THEN 4
            WHEN '65+'   THEN 5
            ELSE 6
        END,
        bucket_share DESC
""")
display(df_channel_age)

### Hallazgos — Customer & Engagement

**1. Adquisición plana sin estacionalidad detectable:**  
El promedio mensual ronda los 68 clientes con stddev ~8. Ningún mes del año
muestra consistentemente mayor o menor adquisición entre los 6 años observados.

En una institución real en LATAM esperaríamos:
- **Enero:** pico por resoluciones de año nuevo y desembolso de bonos
- **Marzo-Abril:** segundo pico pre-Semana Santa
- **Noviembre-Diciembre:** pico por campañas de fin de año
- **Julio-Agosto:** valle por vacaciones

La ausencia de estos patrones es un artefacto del generador que asignó
`registration_date` de forma aleatoria uniforme entre 2020 y 2026.

**2. Sin tendencia interanual de crecimiento:**  
Los totales anuales son muy similares (~810-840 nuevos clientes por año),
sin la curva de crecimiento acelerado que caracterizaría a un fintech en expansión.

**3. El artefacto de corte de mayo 2026:**  
Solo 17 clientes en mayo 2026 vs el promedio de 68. No es una caída real sino
el artefacto del momento de generación del dataset — motivó el fix de
`date_trunc('month', current_date)` en los marts de revenue para excluir el mes
en curso de las agregaciones temporales.

**4. Adopción digital uniforme por segmento (~74%):**  
En una banca real, `private_banking` tendría menor adopción digital y `retail`
joven tendría mayor adopción mobile. La uniformidad es un artefacto del generador.

**Recomendación (contexto real):** La estrategia de marketing debería evaluarse
en función de los canales de preferencia por edad antes que por mes del año.
Los clientes de 18-35 muestran preferencia digital que justifica inversión
en onboarding mobile-first.

---
# Resumen Ejecutivo

## Hallazgos consolidados por página de dashboard

| Dashboard | Pregunta | Hallazgo principal | Limitación dataset |
|---|---|---|---|
| **Revenue & Transactions** | ¿Qué canal es más confiable? | Fallo ~25% en todos los canales · Volúmenes homogéneos (~17.5K tx/canal) | Estados distribuidos uniformemente (real < 3% fallo) |
| **Risk & Credit** | ¿Dónde se concentra el riesgo? | Delinquency ~63% uniforme · 46% clientes en banda Poor · FICO no predice fallo | Sin correlación segmento-riesgo ni utilización-delinquencia |
| **Customer & Engagement** | ¿Hay estacionalidad en adquisición? | Adquisición plana ~68 clientes/mes · Sin patrón estacional · Sin crecimiento interanual | Fechas asignadas aleatoriamente por el generador |

## Conclusión metodológica

Los tres hallazgos confirman la conclusión documentada en `decisions.md`:
el generador sintético distribuyó las variables financieras **sin correlación
entre dimensiones**, produciendo métricas estadísticamente demasiado uniformes
para ser reales.

El valor del pipeline no está en los insights de negocio (limitados por el
dataset sintético) sino en demostrar que:
1. La arquitectura medallion (bronze → silver_raw → silver → gold) escala correctamente en Databricks Free Edition
2. Los 434 tests de dbt detectarían anomalías en datos reales de producción
3. Los 18 marts responden las 24 preguntas de negocio del brief sin joins adicionales desde Power BI
4. La migración Postgres → Databricks preserva la semántica del pipeline con cambios mínimos y documentados en `decisions.md`